In [1]:
import os
import sys
import logging
import numpy as np
import pandas as pd
import joblib
import xgboost as xgb
from sklearn.metrics import roc_auc_score
from sqlalchemy import create_engine
# Add the parent directory to sys.path to find utils
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))


import optuna
from optuna.samplers import TPESampler

/user/rirg2545/miniconda3/envs/hypotension/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Dataset caching for faster multiple training

In [2]:
# Global Cache
DF_FEATURES_CACHE = None
SPLITS_CACHE = None
FEATURE_COLS_CACHE = None
DMATRICES_CACHE = None

### Config

In [3]:
HR_MODEL_UC = "YOUR-PATH/actionable-hypotension/models_hr_extended_uc/xgb_mix_hr.json"
HR_MODEL_C = "YOUR-PATH/actionable-hypotension/models_hr_extended_c/xgb_mix_hr_calibrator.pkl"
BASE_MODEL_C = "YOUR-PATH/actionable-hypotension/extended_evaluation_review/models_calibrated_unbundled/xgb_mix_calibrator.pkl"
BASE_MODEL_UC ="YOUR-PATH/actionable-hypotension/models_given/uncalibrated/xgb_mix.json"
MODELS_DIR = "YOUR-PATH/actionable-hypotension/models_hr_extended_uc"
DASHBOARDS_DIR = "YOUR-PATH/actionable-hypotension/training/optuna_dashboards"
LOG_LEVEL = logging.INFO
DATABASE_URI = "postgresql+psycopg2://USER@localhost:5434/mimic"
engine = create_engine(DATABASE_URI, future=True)

In [4]:

# ----------------------
# Set up logging
# ----------------------
logging.basicConfig(
    level=LOG_LEVEL,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)

### Help Functions : DAS DURCHGEHEN 

In [5]:
def load_data(table_name: str) -> pd.DataFrame:
    """
    Load data from the specified SQL table.
    """
    query = f"SELECT * FROM evaluation.{table_name}"
    df = pd.read_sql(query, engine)
    logging.info(f"Loaded {len(df)} rows, {df.shape[1]} columns")
    return df

# angepasst um treatment_count anzupassen
def preprocess_data(
    df: pd.DataFrame,
    drop_treatment_count: bool = False
) -> pd.DataFrame:
    """
    Drop metadata/time columns and encode the label.
    Optionally drop or binarize the treatment_count column.
    """
    drop_cols = [
        "subject_id", "icustay_id",
        "context_start", "context_end"
    ]
    logging.info("Dropping metadata/time/JSON columns")
    df_features = df.drop(columns=[c for c in drop_cols if c in df.columns])

    # Drop or binarize treatment_count if requested
    if drop_treatment_count and "treatment_count" in df_features.columns:
        logging.info("Dropping treatment_count column")
        df_features = df_features.drop(columns=["treatment_count"])
    # elif binarize_treatment_count and "treatment_count" in df_features.columns:
    #     logging.info("Binarizing treatment_count column")
    #     df_features["treatment_count"] = (df_features["treatment_count"] > 0).astype(int)

    # Label as int
    df_features["label"] = df_features["positive_event"].astype(int)
    if "treatment_given" in df_features.columns:
        df_features = df_features.drop(columns=["treatment_given"])
    if "only_2_values" in df_features.columns:
        df_features = df_features.drop(columns=["only_2_values"])
    if "hr_only_2_values" in df_features.columns:
        df_features = df_features.drop(columns=["hr_only_2_values"])
        
    return df_features


def split_data(df_features: pd.DataFrame) -> dict:
    """
    Split the DataFrame into train, validation, and test sets.
    """
    logging.info("Splitting data into train/val/test")
    return {
        "train": df_features[df_features["split"] == "train"].sample(frac=1, random_state=42).reset_index(drop=True),
        "val":   df_features[df_features["split"] == "val"],
        "test":  df_features[df_features["split"] == "test"],
    }


def get_feature_cols(df_features: pd.DataFrame) -> list:
    """
    Identify feature columns (exclude label and split markers).
    """
    excluded = {"positive_event", "positive_sample", "split", "label"}
    feature_cols = [c for c in df_features.columns if c not in excluded]
    
    # Log the remaining feature columns
    logging.info(f"Using {len(feature_cols)} feature columns: {feature_cols}")
    
    return feature_cols


def create_dmatrices(splits: dict, feature_cols: list) -> dict:
    """
    Create XGBoost DMatrix objects for train, val, and test.
    """
    dmatrices = {}
    for name, subset in splits.items():
        X = subset[feature_cols]
        y = subset["label"]
        logging.info(f"Creating DMatrix for {name} ({len(subset)} rows)")
        dmatrices[name] = xgb.DMatrix(X, label=y, missing=np.nan)
    return dmatrices


def train_model(dtrain, dval, params: dict) -> xgb.Booster:
    """
    Train an XGBoost model with early stopping on the validation set.
    """
    watchlist = [(dtrain, "train"), (dval, "val")]
    logging.info("Starting training")
    bst = xgb.train(
        params,
        dtrain,
        num_boost_round=1000,
        early_stopping_rounds=50,
        evals=watchlist,
        verbose_eval=10,
    )
    return bst


def evaluate_model(bst: xgb.Booster, dtest, y_test: pd.Series) -> float:
    """
    Evaluate the trained model on the test set and log AUC.
    """
    logging.info("Evaluating on test set")
    preds = bst.predict(dtest)
    auc = roc_auc_score(y_test, preds)
    logging.info(f"Test AUC: {auc:.4f}")
    return auc


def save_model(bst: xgb.Booster, model_name: str) -> str:
    """
    Save the model to the models directory using modern JSON format.
    """
    os.makedirs(MODELS_DIR, exist_ok=True)
    
    model_path = os.path.join(MODELS_DIR, f"xgb_{model_name}.json")
    logging.info(f"Saving model to {model_path}")
    
    bst.save_model(model_path)
    return model_path

In [6]:
def prepare_training_data_cached(table_name: str, drop_treatment_given=True, drop_only_2_values=True):
    global DF_FEATURES_CACHE, SPLITS_CACHE, FEATURE_COLS_CACHE, DMATRICES_CACHE

    if all(v is not None for v in [DF_FEATURES_CACHE, SPLITS_CACHE, FEATURE_COLS_CACHE, DMATRICES_CACHE]):
        logging.info("Using cached training data.")
        return DF_FEATURES_CACHE, SPLITS_CACHE, FEATURE_COLS_CACHE, DMATRICES_CACHE

    logging.info(f"Loading training data from ce_approach.{table_name}")
    df = load_data(table_name)

    # Vorverarbeitung
    df_features = preprocess_data(
        df,
        drop_treatment_count=True
    )

    splits = split_data(df_features)
    feature_cols = get_feature_cols(df_features)
    dmatrices = create_dmatrices(splits, feature_cols)

    logging.info("✅ Training data prepared.")
    return df_features, splits, feature_cols, dmatrices

### Hyperparameter Optimization on mixed dateset

In [ ]:
def run_optuna_tuning(table_name: str, model_name: str, dashboard_name: str="xgboost_optimization", n_trials: int = 50, drop_treatment_given=False, drop_only_2_values=False):
    """
    Run Optuna optimization for the given table.
    """
    logging.info(f"Starting Optuna tuning on table: {table_name}")

    os.makedirs(DASHBOARDS_DIR, exist_ok=True)

    # Load and prepare data (with caching)
    df_features, splits, feature_cols, dmatrices = prepare_training_data_cached(table_name, drop_treatment_given=drop_treatment_given, drop_only_2_values=drop_only_2_values)

    dtrain = dmatrices["train"]
    dval   = dmatrices["val"]
    y_val  = splits["val"]["label"]

    # Compute scale_pos_weight for class imbalance
    y_train = splits["train"]["label"]
    scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

    def objective(trial):
        params = {
            "nthread": 16,
            "objective": "binary:logistic",
            "eval_metric": "auc",
            "verbosity": 0,
            "tree_method": "hist",
            "max_bin": 512,
            "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.3, log=True),
            "max_depth": trial.suggest_int("max_depth", 3, 15),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
            "gamma": trial.suggest_float("gamma", 0, 10),
            "subsample": trial.suggest_float("subsample", 0.3, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 1.0),
            "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.3, 1.0),
            "colsample_bynode": trial.suggest_float("colsample_bynode", 0.3, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
            "max_delta_step": trial.suggest_int("max_delta_step", 0, 10),
            "n_estimators": trial.suggest_int("n_estimators", 100, 1000)
        }

        model = xgb.train(
            params,
            dtrain,
            num_boost_round=500,
            early_stopping_rounds=30,
            evals=[(dval, "val")],
            verbose_eval=False
        )

        y_pred_val = model.predict(dval)
        score = roc_auc_score(y_val, y_pred_val)
        return score

    study_name = f"{model_name}"
    storage_path = f"sqlite:///{DASHBOARDS_DIR}/{dashboard_name}.db"

    study = optuna.create_study(
        study_name=study_name,
        direction="maximize",
        sampler=TPESampler(seed=42),
        storage=storage_path,
        load_if_exists=True
    )
    study.optimize(objective, n_trials=n_trials)

    logging.info(f"Best score: {study.best_value:.5f}")
    logging.info(f"Best params: {study.best_params}")

    # Retrain best model on full training set
    best_params = study.best_params
    best_params.update({
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "tree_method": "hist",
        "max_bin": 512,
        "verbosity": 1
    })

    bst = xgb.train(
        best_params,
        dtrain,
        num_boost_round=500,
        early_stopping_rounds=30,
        evals=[(dval, "val")],
        verbose_eval=10
    )

    # Save model
    
    os.makedirs(MODELS_DIR, exist_ok=True)
    model_path = os.path.join(MODELS_DIR, f"xgb_{model_name}.model")
    save_model(bst, model_name)
    logging.info(f"Saved best model to: {model_path}")

    return bst, study


In [15]:
bst, study = run_optuna_tuning("merged_mix_features", model_name="mix", n_trials=200, drop_treatment_given=True, drop_only_2_values=True)

2026-04-20 10:46:06 [INFO] Starting Optuna tuning on table: merged_mix_features
2026-04-20 10:46:06 [INFO] Loading training data from ce_approach.merged_mix_features


ProgrammingError: (psycopg2.errors.UndefinedTable) relation "evaluation.merged_mix_features" does not exist
LINE 1: SELECT * FROM evaluation.merged_mix_features
                      ^

[SQL: SELECT * FROM evaluation.merged_mix_features]
(Background on this error at: https://sqlalche.me/e/20/f405)

### Train mix, invasive and non-invasive models with optimized hyperparameters

In [7]:
def train_model_for_table(table_name: str, model_name: str, override_params: dict = None, drop_treatment_given=True, drop_only_2_values=True) -> xgb.Booster:
    df_features, splits, feature_cols, dmatrices = prepare_training_data_cached(table_name, drop_treatment_given=drop_treatment_given, drop_only_2_values=drop_only_2_values)
    
    y_train = splits["train"]["label"]
    neg = (y_train == 0).sum()
    pos = (y_train == 1).sum()
    scale_pos_weight = neg / pos
    logging.info(f"Calculated scale_pos_weight: {scale_pos_weight:.2f}")

    default_params = {
        "objective":     "binary:logistic",
        "eval_metric":   "auc",
        "tree_method":   "hist",
        "learning_rate": 0.01,
        "max_depth":     10,
        "verbosity":     1,
        "scale_pos_weight": scale_pos_weight
    }

    params = override_params or default_params

    bst = train_model(dmatrices["train"], dmatrices["val"], params)
    evaluate_model(bst, dmatrices["test"], splits["test"]["label"])
    save_model(bst, model_name)

    return bst

In [8]:
def load_and_prepare_validation_data(table_name, table_source, drop_treatment_given=False, drop_only_2_values=False):
    
    
    DATABASE_URI = "postgresql+psycopg2://USER@localhost:5434/mimic"
    engine = create_engine(DATABASE_URI, future=True)
    df = pd.read_sql(f"""
        SELECT * FROM {table_source}.{table_name}
        WHERE split IN ('val', 'test')
    """, engine)

    drop_cols = ["subject_id", "icustay_id", "context_start", "context_end", "hr_only_2_values"]
    df = df.drop(columns=[c for c in drop_cols if c in df.columns])
    
    
    df["label"] = df["positive_event"].astype(int)

    excluded = {"positive_event", "positive_sample", "split", "label"}
    
    if drop_treatment_given:
        df = df.drop(columns=["treatment_given"])
    if drop_only_2_values:
        df = df.drop(columns=["only_2_values"])
        
    
    feature_cols = [c for c in df.columns if c not in excluded]

    val_df  = df[df["split"] == "val"]
    test_df = df[df["split"] == "test"]

    x_val = val_df[feature_cols]
    y_val = val_df["label"]

    return x_val, y_val

In [9]:
def bootstrap_auc(y_true, y_pred, n_bootstrap=1000, ci=0.95, seed=42):
    """
    Returns (auc, lower, upper) using percentile bootstrap.
    """
    rng = np.random.default_rng(seed)
    aucs = []
    n = len(y_true)
    y_true_arr = np.array(y_true)
    y_pred_arr = np.array(y_pred)

    for _ in range(n_bootstrap):
        idx = rng.integers(0, n, size=n)
        y_t = y_true_arr[idx]
        y_p = y_pred_arr[idx]
        if len(np.unique(y_t)) < 2:
            continue  # skip degenerate bootstrap samples
        aucs.append(roc_auc_score(y_t, y_p))

    alpha = (1 - ci) / 2
    lower = np.percentile(aucs, 100 * alpha)
    upper = np.percentile(aucs, 100 * (1 - alpha))
    point = roc_auc_score(y_true_arr, y_pred_arr)
    return point, lower, upper

In [12]:
def load_and_evaluate_model(table_name, table_source, model_path, calibrator_path):
    # --- Load data ---
    x_val, y_val = load_and_prepare_validation_data(table_name, table_source, True, True)

    # --- Load XGBoost model ---
    booster = xgb.Booster()
    booster.load_model(model_path)

    # --- Load calibrator ---
    iso = joblib.load(calibrator_path)

    dval = xgb.DMatrix(x_val)

    # Raw probabilities from model
    y_pred_raw = booster.predict(dval)

    # Calibrated probabilities
    y_pred_cal = iso.transform(y_pred_raw)

    
    auc_cal = bootstrap_auc(y_val, y_pred_cal)

    print(f"({model_path}) AUC calibrated: {auc_cal}")

    return {
        "auc_calibrated": auc_cal
    }

In [16]:
from sklearn.metrics import roc_curve
import numpy as np

def find_threshold_for_sensitivity(y_true, y_pred_proba, target_sensitivity=0.80):
    fpr, tpr, thresholds = roc_curve(y_true, y_pred_proba)

    # Find all thresholds where sensitivity >= target
    valid_idxs = np.where(tpr >= target_sensitivity)[0]

    if len(valid_idxs) == 0:
        raise ValueError("No threshold achieves the desired sensitivity")

    # Choose the threshold with the highest specificity (lowest FPR)
    best_idx = valid_idxs[np.argmin(fpr[valid_idxs])]

    return thresholds[best_idx]

In [19]:
def load_and_evaluate_model_extended(table_name, table_source, model_path, calibrator_path):
    import numpy as np
    from sklearn.metrics import confusion_matrix

    # --- Load data ---
    x_val, y_val = load_and_prepare_validation_data(table_name, table_source, True, True)

    # --- Load XGBoost model ---
    booster = xgb.Booster()
    booster.load_model(model_path)

    # --- Load calibrator ---
    iso = joblib.load(calibrator_path)

    dval = xgb.DMatrix(x_val)

    # Raw probabilities from model
    y_pred_raw = booster.predict(dval)

    # Calibrated probabilities
    y_pred_cal = iso.transform(y_pred_raw)

    # --- AUC ---
    auc_cal = bootstrap_auc(y_val, y_pred_cal)

    # --- Thresholding ---
    threshold = find_threshold_for_sensitivity(y_val,y_pred_cal)
    y_pred_bin = (y_pred_cal >= threshold).astype(int)

    # --- Confusion matrix ---
    tn, fp, fn, tp = confusion_matrix(y_val, y_pred_bin).ravel()

    # --- Derived metrics ---
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    precision   = tp / (tp + fp) if (tp + fp) > 0 else 0.0

    print(f"({model_path}) AUC calibrated: {auc_cal}")
    print(f"Threshold: {threshold}")
    print(f"TP: {tp}, FP: {fp}, TN: {tn}, FN: {fn}")
    print(f"Sensitivity (Recall): {sensitivity:.4f}")
    print(f"Specificity: {specificity:.4f}")
    print(f"Precision: {precision:.4f}")

    return {
        "auc_calibrated": auc_cal,
        "threshold": threshold,
        "tp": int(tp),
        "fp": int(fp),
        "tn": int(tn),
        "fn": int(fn),
        "sensitivity": sensitivity,
        "specificity": specificity,
        "precision": precision
    }

In [22]:
#1) base: 
load_and_evaluate_model("merged_mix_features", "ce_approach", BASE_MODEL_UC, BASE_MODEL_C)

(YOUR-PATH/actionable-hypotension/models_given/uncalibrated/xgb_mix.json) AUC calibrated: (0.8273572644168876, np.float64(0.8185264239188337), np.float64(0.8367271889881779))


{'auc_calibrated': (0.8273572644168876,
  np.float64(0.8185264239188337),
  np.float64(0.8367271889881779))}

In [13]:
load_and_evaluate_model("merged_mix_hr_dataset", "evaluation", HR_MODEL_UC, HR_MODEL_C)

(YOUR-PATH/actionable-hypotension/models_hr_extended_uc/xgb_mix_hr.json) AUC calibrated: (0.797351764263968, np.float64(0.7879008050351753), np.float64(0.8068073051902179))


{'auc_calibrated': (0.797351764263968,
  np.float64(0.7879008050351753),
  np.float64(0.8068073051902179))}

In [21]:
load_and_evaluate_model_extended("merged_mix_hr_dataset", "evaluation", HR_MODEL_UC, HR_MODEL_C)

(YOUR-PATH/actionable-hypotension/models_hr_extended_uc/xgb_mix_hr.json) AUC calibrated: (0.797351764263968, np.float64(0.7879008050351753), np.float64(0.8068073051902179))
Threshold: 0.0013438735622912645
TP: 1568, FP: 404957, TN: 718808, FN: 388
Sensitivity (Recall): 0.8016
Specificity: 0.6396
Precision: 0.0039


{'auc_calibrated': (0.797351764263968,
  np.float64(0.7879008050351753),
  np.float64(0.8068073051902179)),
 'threshold': np.float32(0.0013438736),
 'tp': 1568,
 'fp': 404957,
 'tn': 718808,
 'fn': 388,
 'sensitivity': np.float64(0.8016359918200409),
 'specificity': np.float64(0.6396426299092782),
 'precision': np.float64(0.003857081360309944)}

In [23]:
load_and_evaluate_model_extended("merged_mix_features", "ce_approach", BASE_MODEL_UC, BASE_MODEL_C)

(YOUR-PATH/actionable-hypotension/models_given/uncalibrated/xgb_mix.json) AUC calibrated: (0.8273572644168876, np.float64(0.8185264239188337), np.float64(0.8367271889881779))
Threshold: 0.0011452179169282317
TP: 1666, FP: 431882, TN: 691883, FN: 290
Sensitivity (Recall): 0.8517
Specificity: 0.6157
Precision: 0.0038


{'auc_calibrated': (0.8273572644168876,
  np.float64(0.8185264239188337),
  np.float64(0.8367271889881779)),
 'threshold': np.float32(0.0011452179),
 'tp': 1666,
 'fp': 431882,
 'tn': 691883,
 'fn': 290,
 'sensitivity': np.float64(0.8517382413087935),
 'specificity': np.float64(0.6156829942203219),
 'precision': np.float64(0.00384271176432598)}

In [ ]:
# Load study from Optuna database
dashboard_name = "xgboost_optimization"
storage = f"sqlite:///{DASHBOARDS_DIR}/{dashboard_name}.db"
study_name = "mix"

study = optuna.load_study(study_name=study_name, storage=storage)
best_trial = study.best_trial

# Extract best hyperparameters and add / override fixed values
params = best_trial.params.copy()
params.update({
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "tree_method": "hist",
    "nthread": 16,
    "max_bin": 512,
    "verbosity": 1,
    "scale_pos_weight": 1
})

params

{'learning_rate': 0.03550532419674665,
 'max_depth': 7,
 'min_child_weight': 13,
 'gamma': 2.770872774283558,
 'subsample': 0.7695776075436386,
 'colsample_bytree': 0.8995644394424454,
 'colsample_bylevel': 0.9287452874368791,
 'colsample_bynode': 0.8520653596079167,
 'reg_alpha': 0.03189148967972523,
 'reg_lambda': 3.5711779652989895e-07,
 'max_delta_step': 3,
 'n_estimators': 194,
 'objective': 'binary:logistic',
 'eval_metric': 'auc',
 'tree_method': 'hist',
 'nthread': 16,
 'max_bin': 512,
 'verbosity': 1,
 'scale_pos_weight': 1}

In [ ]:
# Round all float hyperparameters to 4 decimals for reproducibility & readability
params = {
    k: round(v, 4) if isinstance(v, float) else v
    for k, v in params.items()
}

params

{'learning_rate': 0.0355,
 'max_depth': 7,
 'min_child_weight': 13,
 'gamma': 2.7709,
 'subsample': 0.7696,
 'colsample_bytree': 0.8996,
 'colsample_bylevel': 0.9287,
 'colsample_bynode': 0.8521,
 'reg_alpha': 0.0319,
 'reg_lambda': 0.0,
 'max_delta_step': 3,
 'n_estimators': 194,
 'objective': 'binary:logistic',
 'eval_metric': 'auc',
 'tree_method': 'hist',
 'nthread': 16,
 'max_bin': 512,
 'verbosity': 1,
 'scale_pos_weight': 1}

In [10]:
train_model_for_table(table_name="merged_mix_hr_dataset", model_name="mix_hr", drop_treatment_given=True, drop_only_2_values=True) # no overriding of params for initial experiment

2026-04-20 21:36:35 [INFO] Loading training data from ce_approach.merged_mix_hr_dataset
2026-04-20 21:38:09 [INFO] Loaded 7257928 rows, 54 columns
2026-04-20 21:38:09 [INFO] Dropping metadata/time/JSON columns
2026-04-20 21:38:10 [INFO] Splitting data into train/val/test
2026-04-20 21:38:15 [INFO] Using 44 feature columns: ['mean', 'median', 'min', 'max', 'std', 'iqr', 'first', 'last', 'rate_change', 'slope', 'weighted_mean', 'gender_bin', 'ethnicity_bin', 'age_bin', 'height_bin', 'weight_bin', 'bmi_bin', 'obesity', 'hypertension', 'diabetes', 'kidney_disease', 'lung_disease', 'heart_disease', 'drug_abuse', 'depression', 'sedatives_given', 'blood_products_transfusions_given', 'antibiotics_given', 'anticoagulants_antiplatelets_given', 'neuromuscular_blockers_given', 'analgesics_given', 'crystalloids_given', 'electrolytes_given', 'gi_protection_given', 'parenteral_nutrition_given', 'antiarrhythmics_given', 'hr_mean', 'hr_median', 'hr_std', 'hr_iqr', 'hr_first', 'hr_last', 'hr_slope', 'hr

[0]	train-auc:0.84540	val-auc:0.73101
[10]	train-auc:0.86220	val-auc:0.76306
[20]	train-auc:0.86884	val-auc:0.76908
[30]	train-auc:0.87644	val-auc:0.77447
[40]	train-auc:0.88150	val-auc:0.77964
[50]	train-auc:0.88525	val-auc:0.78236
[60]	train-auc:0.88885	val-auc:0.78522
[70]	train-auc:0.89203	val-auc:0.78763
[80]	train-auc:0.89491	val-auc:0.78942
[90]	train-auc:0.89739	val-auc:0.79048
[100]	train-auc:0.90010	val-auc:0.79107
[110]	train-auc:0.90295	val-auc:0.79156
[120]	train-auc:0.90539	val-auc:0.79247
[130]	train-auc:0.90779	val-auc:0.79330
[140]	train-auc:0.91025	val-auc:0.79368
[150]	train-auc:0.91280	val-auc:0.79397
[160]	train-auc:0.91515	val-auc:0.79437
[170]	train-auc:0.91758	val-auc:0.79509
[180]	train-auc:0.91982	val-auc:0.79515
[190]	train-auc:0.92175	val-auc:0.79519
[200]	train-auc:0.92373	val-auc:0.79564
[210]	train-auc:0.92553	val-auc:0.79598
[220]	train-auc:0.92752	val-auc:0.79628
[230]	train-auc:0.92944	val-auc:0.79655
[240]	train-auc:0.93129	val-auc:0.79656
[250]	train

2026-04-20 21:39:34 [INFO] Evaluating on test set
2026-04-20 21:39:34 [INFO] Test AUC: 0.7972
2026-04-20 21:39:34 [INFO] Saving model to YOUR-PATH/actionable-hypotension/models_hr_extended_uc/xgb_mix_hr.json
